In [1]:
import os
import warnings
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from gensim.models import Word2Vec

In [2]:
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    
    warnings.filterwarnings("ignore")
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)
    
    print(f"Random seed set to {seed}")

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2 ** 32
    random.seed(worker_seed)
    np.random.seed(worker_seed)

In [3]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Random seed set to 42
Device: cuda


In [4]:
INPUT_ROOT = "/kaggle/input/datasets/kiennguyenvutrung/input-df26-round3"
WORK_DIR = "/kaggle/working"

X_TRAIN = f"{INPUT_ROOT}/X_train.csv"
X_VAL = f"{INPUT_ROOT}/X_val.csv"
X_TEST = f"{INPUT_ROOT}/X_test.csv"
Y_TRAIN = f"{INPUT_ROOT}/Y_train.csv"
Y_VAL = f"{INPUT_ROOT}/Y_val.csv"

SUBMISSION_FILE = f"{WORK_DIR}/submission.csv"
CHECKPOINT_FILE = f"{WORK_DIR}/best_model.pth"

In [5]:
SEQ_LENGTH = 66
FEATURE_COLS = [f"feature_{i}" for i in range(1, SEQ_LENGTH + 1)]

ATTRIBUTE_LIST = [1, 2, 3, 4, 5, 6]
ATTRIBUTE_COLS = ["start_year", "attr_1", "attr_2", "attr_3",
                  "end_year", "attr_4", "attr_5", "attr_6"]

EMBEDDING_DIM = 256
DILATIONS = [1, 2, 4, 8]
DROPOUT_RATE = 0.3
HEADS = 4
WINDOW_SIZE = 3

BATCH_SIZE = 1024
NUM_EPOCHS = 100
EARLY_STOPPING = 15
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-2
MAX_LR = 1e-3
PCT_START = 0.2
FINAL_DIV_FACTOR = 10000.0

M_LIST_LOSS = [1.0, 12.0, 31.0, 99.0, 1.0, 12.0, 31.0, 99.0]
M_LIST_METRIC = [12.0, 31.0, 99.0, 12.0, 31.0, 99.0]
W_LIST_VALS = [1.0, 1.0, 100.0, 1.0, 1.0, 100.0]

DAYS_IN_MONTH = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
DAYS_MAP = {
    1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
    7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
}

In [6]:
def drop_duplicates_custom(x_df, y_df, feature_cols=FEATURE_COLS, name="Train"):
    len_before = len(x_df)
    keep_idx = x_df.drop_duplicates(subset=feature_cols, keep="first").index
    
    x_df = x_df.loc[keep_idx].reset_index(drop=True)
    y_df = y_df.loc[keep_idx].reset_index(drop=True)
    
    print(f" - Loại bỏ {len_before - len(x_df)} hàng trùng lặp trong {name}")
    return x_df, y_df

In [7]:
def drop_overlap_train_val(x_train, y_train, x_val, feature_cols=FEATURE_COLS):
    len_before = len(x_train)
    
    val_features_unique = x_val[feature_cols].drop_duplicates().copy()
    val_features_unique["is_in_val"] = True
    
    x_train["old_idx"] = x_train.index
    train_merged = x_train.merge(val_features_unique, on=feature_cols, how="left")
    
    clean_train_idx = train_merged[train_merged["is_in_val"].isna()]["old_idx"]
    
    x_train = x_train.loc[clean_train_idx].drop(columns=["old_idx"]).reset_index(drop=True)
    y_train = y_train.loc[clean_train_idx].reset_index(drop=True)
    
    print(f" - Loại bỏ {len_before - len(x_train)} chuỗi Train bị trùng với Val")
    return x_train, y_train

In [8]:
def validate_and_clean_dates(x_df, y_df, days_map=DAYS_MAP, name="Train"):
    max_days_start = y_df["attr_1"].map(days_map)
    invalid_start_day_mask = y_df["attr_2"] > max_days_start

    max_days_end = y_df["attr_4"].map(days_map)
    invalid_end_day_mask = y_df["attr_5"] > max_days_end

    all_invalid_mask = invalid_start_day_mask | invalid_end_day_mask
    invalid_count = all_invalid_mask.sum()

    if invalid_count > 0:
        print(f" - Phát hiện {invalid_count} hàng có lỗi ngày trong {name}")
        
        y_df.loc[invalid_start_day_mask, "attr_2"] = max_days_start[invalid_start_day_mask]
        y_df.loc[invalid_end_day_mask, "attr_5"] = max_days_end[invalid_end_day_mask]
        
    else:
        print(f" - Không có lỗi ngày tháng trong {name}")

    return x_df, y_df

In [9]:
def get_vocab_size(x_train, x_val, x_test, feature_cols=FEATURE_COLS):
    all_x_data = pd.concat([x_train[feature_cols], x_val[feature_cols]], axis=0)
    unique_ids = pd.unique(all_x_data.values.ravel())
    unique_ids = [uid for uid in unique_ids if not pd.isna(uid) and uid != 0]
    
    id_to_index = {0: 0, "UNK": 1}
    for idx, raw_id in enumerate(sorted(unique_ids), start=2):
        id_to_index[raw_id] = idx
        
    vocab_size = max(id_to_index.values()) + 1
    print(f" - Kích thước từ điển {vocab_size} (PAD=0, UNK=1)")

    map_func = np.vectorize(lambda x: id_to_index.get(x, 1))
    
    for df in [x_train, x_val, x_test]:
        df.loc[:, feature_cols] = map_func(df[feature_cols].values)

    return x_train, x_val, x_test, vocab_size

In [10]:
def manual_augment(x_train, y_train, x_test, feature_cols):
    train_lengths = (x_train[feature_cols] != 0).sum(axis=1)
    test_lengths = (x_test[feature_cols] != 0).sum(axis=1)
    
    train_counts = train_lengths.value_counts()
    test_counts = test_lengths.value_counts()
    
    augmented_x = [x_train]
    augmented_y = [y_train]
    
    for length, c_test in test_counts.items():
        c_train = train_counts.get(length, 0)

        if c_test > c_train:
            if c_train == 0:
                multiplier = 10
            else:
                multiplier = min(c_test / c_train, 10)
            
            n_repeats = int(multiplier) - 1
            
            if n_repeats > 0:
                mask = (train_lengths == length)
                x_to_dup = x_train[mask]
                y_to_dup = y_train[mask]
                
                if len(x_to_dup) > 0:
                    for _ in range(n_repeats):
                        augmented_x.append(x_to_dup)
                        augmented_y.append(y_to_dup)
    x_train_new = pd.concat(augmented_x, ignore_index=True)
    y_train_new = pd.concat(augmented_y, ignore_index=True)
    
    idx = np.random.permutation(len(x_train_new))
    x_train_new = x_train_new.iloc[idx].reset_index(drop=True)
    y_train_new = y_train_new.iloc[idx].reset_index(drop=True)
    
    print(f" - Train tăng từ {len(x_train)} lên {len(x_train_new)} mẫu")
    return x_train_new, y_train_new

In [11]:
def process_data(x_train, y_train, x_val, y_val, x_test,
                 feature_cols=FEATURE_COLS,
                 attr_cols=ATTRIBUTE_COLS,
                 days_map=DAYS_MAP):
    
    for df in [x_train, x_val, x_test]:
        df[feature_cols] = df[feature_cols].fillna(0).astype(np.int64)
    for df in [y_train, y_val]:
        df["start_year"] = 0
        df["end_year"] = (df["attr_4"] < df["attr_1"]).astype(np.int64)
        df[attr_cols] = df[attr_cols].fillna(0).astype(np.float32)

    print("Xử lý trùng lặp nội bộ...")
    x_train, y_train = drop_duplicates_custom(x_train, y_train, feature_cols, "Train")
    x_val, y_val = drop_duplicates_custom(x_val, y_val, feature_cols, "Val")
    
    print("Xử lý trùng lặp giữa Train và Val...")
    x_train, y_train = drop_overlap_train_val(x_train, y_train, x_val, feature_cols)

    print("Xử lý lỗi ngày tháng...")
    x_train, y_train = validate_and_clean_dates(x_train, y_train, days_map, "Train")
    x_val, y_val = validate_and_clean_dates(x_val, y_val, days_map, "Val")

    print("Xây dựng từ điển và map index...")
    x_train, x_val, x_test, vocab_size = get_vocab_size(x_train, x_val, x_test, feature_cols)

    print("Sinh thủ công thêm chuỗi...")
    x_train, y_train = manual_augment(x_train, y_train, x_test, feature_cols)
    
    return x_train, y_train, x_val, y_val, x_test, vocab_size

In [12]:
class UserBehaviorDataset(Dataset):
    def __init__(self, x_df, y_df=None, 
                 feature_cols=FEATURE_COLS, attr_cols=ATTRIBUTE_COLS, augment=False):
        self.x_data = x_df[feature_cols].values
        self.augment = augment 
        
        self.has_labels = y_df is not None
        if self.has_labels:
            self.y_data = y_df[attr_cols].values

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        x_tensor = torch.tensor(self.x_data[idx], dtype=torch.long)
        
        if self.augment:
            actions = x_tensor[x_tensor != 0]
            n_actions = len(actions)
            total_slots = len(x_tensor) 
            
            if torch.rand(1).item() < 0.5:
                n_mask = (n_actions // 10) + 1
                possible_indices = torch.arange(n_actions - 1)
                
                if len(possible_indices) >= n_mask:
                    mask_indices = possible_indices[torch.randperm(len(possible_indices))[:n_mask]]
                    actions[mask_indices] = 0
            
            if torch.rand(1).item() < 0.5:
                new_indices = torch.randperm(total_slots)[:n_actions].sort()[0]
                dilated_x = torch.zeros_like(x_tensor)
                dilated_x[new_indices] = actions
                x_tensor = dilated_x
            else:
                new_x = torch.zeros_like(x_tensor)
                new_x[:n_actions] = actions
                x_tensor = new_x

        if self.has_labels:
            y_tensor = torch.tensor(self.y_data[idx], dtype=torch.float32)
            return x_tensor, y_tensor
        
        return x_tensor

In [13]:
def create_dataloaders(x_train, y_train, x_val, y_val, x_test, 
                       batch_size=BATCH_SIZE,
                       num_workers=2,
                       seed_worker=seed_worker,
                       data_generator=data_generator):
    
    train_dataset = UserBehaviorDataset(x_train, y_train, augment=True)
    val_dataset = UserBehaviorDataset(x_val, y_val, augment=False)
    test_dataset = UserBehaviorDataset(x_test, None, augment=False)

    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers, 
        worker_init_fn=seed_worker, 
        generator=data_generator,
        pin_memory=True, 
        drop_last=True, 
        persistent_workers=True  
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,           
        pin_memory=True,
        persistent_workers=False 
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,
        pin_memory=True
    )
    
    return train_loader, val_loader, test_loader

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionPooling1D(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.attn = nn.Linear(in_features, 1)

    def forward(self, x, mask):
        scores = self.attn(x) 
        if mask is not None:
            fill_value = torch.finfo(scores.dtype).min
            scores = scores.masked_fill(~mask.unsqueeze(-1), fill_value)
        attn_weights = F.softmax(scores, dim=1)
        return torch.sum(x * attn_weights, dim=1)

class CascadeRegressionHead(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.head_start = nn.Sequential(
            nn.Linear(d_model, 64), nn.GELU(), nn.Linear(64, 3)
        )
        self.start_proj = nn.Sequential(
            nn.Linear(3, 16), nn.GELU(), nn.Linear(16, 32)
        )
        
        self.head_factory = nn.Sequential(
            nn.Linear(d_model + 32, 128), nn.LayerNorm(128), nn.GELU(), 
            nn.Dropout(0.2), nn.Linear(128, 64), nn.GELU(), nn.Linear(64, 2)
        )
        
        self.head_end = nn.Sequential(
            nn.Linear(d_model + 32, 64), nn.GELU(), nn.Linear(64, 3)
        )

    def forward(self, x):
        start_preds = self.head_start(x)  
        start_context = self.start_proj(start_preds)
        
        factory_in = torch.cat([x, start_context], dim=-1)
        factory_preds = self.head_factory(factory_in) 
        
        end_in = torch.cat([x, start_context], dim=-1)
        end_preds = self.head_end(end_in)    
        
        s_year = torch.sigmoid(start_preds[:, 0])
        e_year = torch.sigmoid(end_preds[:, 0])
        s_month = 1.0 + 11.0 * torch.sigmoid(start_preds[:, 1])
        e_month = 1.0 + 11.0 * torch.sigmoid(end_preds[:, 1])
        s_day = 1.0 + 30.0 * torch.sigmoid(start_preds[:, 2])
        e_day = 1.0 + 30.0 * torch.sigmoid(end_preds[:, 2])
        s_power = 99.0 * torch.sigmoid(factory_preds[:, 0])
        e_power = 99.0 * torch.sigmoid(factory_preds[:, 1])
        
        return [s_year, s_month, s_day, s_power, e_year, e_month, e_day, e_power]

class GCE_FusionLayer(nn.Module):
    def __init__(self, vocab_size, d_model, dropout=0.2):
        super().__init__()
        self.global_memory = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.fusion_gate = nn.Linear(d_model * 2, d_model)
        self.transform = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, local_features, input_ids):
        global_features = self.global_memory(input_ids) 
        concat_features = torch.cat([local_features, global_features], dim=-1)
        gate_score = torch.sigmoid(self.fusion_gate(concat_features)) 
        fused_features = gate_score * local_features + (1.0 - gate_score) * global_features
        out = self.dropout(F.gelu(self.transform(fused_features)))
        return self.norm(local_features + out) 

class ResidualBiGRUBlock(nn.Module):
    def __init__(self, d_model, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        res = x
        out, _ = self.gru(x)
        out = self.dropout(out)
        return self.norm(res + out)

class MultiScaleConv1D(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        dim = d_model // 4
        self.conv1 = nn.Conv1d(d_model, dim, kernel_size=1)
        self.conv3 = nn.Conv1d(d_model, dim, kernel_size=3, padding=1)
        self.conv5 = nn.Conv1d(d_model, dim, kernel_size=5, padding=2)
        # Bắt chước dilation của GNN
        self.conv_dilated = nn.Conv1d(d_model, dim, kernel_size=3, padding=2, dilation=2) 
        self.proj = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x_t = x.transpose(1, 2) 
        c1 = F.gelu(self.conv1(x_t))
        c3 = F.gelu(self.conv3(x_t))
        c5 = F.gelu(self.conv5(x_t))
        cd = F.gelu(self.conv_dilated(x_t))
        
        out = torch.cat([c1, c3, c5, cd], dim=1).transpose(1, 2)
        return self.norm(x + self.proj(out))

class AdvancedTARNet(nn.Module):
    def __init__(self, vocab_size, seq_length=66, d_model=256, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_embedding = nn.Embedding(seq_length, d_model)
        
        self.multi_scale_conv = MultiScaleConv1D(d_model)
        self.conv_dropout = nn.Dropout(dropout)
        
        self.rnn_blocks = nn.ModuleList([
            ResidualBiGRUBlock(d_model, dropout) for _ in range(2)
        ])
        
        self.gce_fusion = GCE_FusionLayer(vocab_size, d_model, dropout)
        
        self.transformer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, dim_feedforward=d_model*2, 
            dropout=dropout, batch_first=True, activation='gelu'
        )
        
        self.attn_pool = AttentionPooling1D(d_model)
        self.dropout = nn.Dropout(dropout)
        self.cascade_head = CascadeRegressionHead(d_model)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.GRU):
            for name, param in m.named_parameters():
                if 'weight_ih' in name:
                    nn.init.xavier_uniform_(param.data)
                elif 'weight_hh' in name:
                    nn.init.orthogonal_(param.data)

    def forward(self, x):
        B, L = x.size()
        mask = (x != 0)
        mask_float = mask.unsqueeze(-1).float()
        
        pos = torch.arange(L, device=x.device).unsqueeze(0).expand(B, L)
        h_emb = (self.embedding(x) + self.pos_embedding(pos)) * mask_float
        
        h_conv = self.multi_scale_conv(h_emb)
        h_conv = self.conv_dropout(h_conv) * mask_float
        
        h_rnn = h_conv
        for rnn_block in self.rnn_blocks:
            h_rnn = rnn_block(h_rnn) * mask_float
            
        h_fuse = self.gce_fusion(h_rnn, x) * mask_float
        
        h_attn = self.transformer(h_fuse, src_key_padding_mask=~mask)
        
        h_final = h_emb + h_conv + h_attn
        
        x_pool = self.dropout(self.attn_pool(h_final, mask))
        return self.cascade_head(x_pool)

In [15]:
class HybridThresholdLoss(nn.Module):
    def __init__(self, device, start_threshold=0.01, end_threshold=0.05):
        super().__init__()
        self.device = device
        self.M = torch.tensor(M_LIST_LOSS, device=device).view(1, -1)
        
        self.start_w = torch.tensor([100.0, 100.0, 100.0, 1.0, 100.0, 100.0, 100.0, 1.0], device=device).view(1, -1)
        
        self.end_w = torch.tensor([0.0, 1.0, 1.0, 100.0, 0.0, 1.0, 1.0, 100.0], device=device).view(1, -1)
        self.w = self.start_w.clone()
        
        self.start_delta = start_threshold
        self.end_delta = end_threshold
        self.delta = start_threshold 

    def update_params(self, current_epoch, total_epochs=10):
        progress = current_epoch / total_epochs
        
        self.w = self.start_w + (self.end_w - self.start_w) * progress
        
        current_delta = self.start_delta + ((self.end_delta - self.start_delta) * progress)
        self.delta = current_delta

        return self.w[0, 0].item(), self.w[0, 1].item(), self.w[0, 3].item()
        
    def forward(self, preds_list, targets):
        if isinstance(preds_list, list):
            preds = torch.stack(preds_list, dim=1)
        else:
            preds = preds_list

        abs_err = torch.abs(targets.float() - preds) / self.M
        quad = torch.clamp(abs_err, max=self.delta)
        lin = abs_err - quad

        hybrid_loss = self.w * (0.5 * (quad ** 2) + self.delta * lin)
        return torch.mean(torch.sum(hybrid_loss, dim=1))

In [16]:
def train_model(model, 
                train_loader, 
                val_loader, 
                y_val_orig, 
                optimizer, 
                scheduler, 
                device=DEVICE):
    
    loss_fn = HybridThresholdLoss(device)
    best_score = float("inf")
    scaler = GradScaler()
    early_count = 0
    
    for epoch in range(NUM_EPOCHS):
        if epoch <= 10:
            loss_fn.update_params(epoch)
        
        model.train()
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            
            with autocast():
                preds = model(x)
                
            loss = loss_fn(preds, y.float())
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        
        y_pred_val = run_inference(model, val_loader, device)
        val_score = evaluate_wmse(y_val_orig, y_pred_val)
        
        print(f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | LR: {optimizer.param_groups[0]['lr']:.6f} | "
              f"Train Loss: {avg_train_loss:.4f} | Val Score: {val_score:.4f}")
        
        if val_score < best_score:
            best_score = val_score
            torch.save(model.state_dict(), CHECKPOINT_FILE)
            print(f"  => Mô hình tốt nhất! (Val Score: {val_score:.4f})")
            early_count = 0
        else:
            early_count += 1
            print(f"  => Mô hình không cải thiện ({early_count}/{EARLY_STOPPING})")
            if early_count >= EARLY_STOPPING:
                print(f"  => Dừng sớm tại Epoch {epoch + 1} (Best Score: {best_score:.4f})")
                break

In [17]:
def post_process_predictions(preds_tensor):
    preds = preds_tensor.clone()
    preds = torch.round(preds)
    
    preds[:, [0, 3]] = (preds[:, [0, 3]] - 1) % 12 + 1
    preds[:, [0, 3]] = torch.clamp(preds[:, [0, 3]], 1, 12)

    days_in_month_t = torch.tensor(DAYS_IN_MONTH, device=preds.device, dtype=preds.dtype)
    
    for col_idx, month_idx in [(1, 0), (4, 3)]:
        m_indices = (preds[:, month_idx].long() - 1).clamp(0, 11)
        max_days = days_in_month_t[m_indices]
        
        preds[:, col_idx] = torch.clamp(preds[:, col_idx], min=1)
        preds[:, col_idx] = torch.min(preds[:, col_idx], max_days)

    preds[:, [2, 5]] = torch.clamp(preds[:, [2, 5]], 0, 99)
    
    return preds

def run_inference(model, loader, device=DEVICE):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            outputs = model(x.to(device))
            all_preds.append(torch.stack(outputs, dim=1) if isinstance(outputs, list) else outputs)
    
    all_preds = torch.cat(all_preds, dim=0)[:, [1, 2, 3, 5, 6, 7]]
    return post_process_predictions(all_preds).cpu().numpy()

def evaluate_wmse(y_true, y_pred):
    M, W = np.array(M_LIST_METRIC), np.array(W_LIST_VALS)
    return np.mean(np.sum(W * np.power((y_true - y_pred) / M, 2), axis=1))

In [18]:
x_train = pd.read_csv(X_TRAIN)
y_train = pd.read_csv(Y_TRAIN)
x_val = pd.read_csv(X_VAL)
y_val = pd.read_csv(Y_VAL)
x_test = pd.read_csv(X_TEST)

x_train, y_train, x_val, y_val, x_test, VOCAB_SIZE = process_data(
    x_train, y_train, x_val, y_val, x_test
)

y_val_orig = (y_val[ATTRIBUTE_COLS].values)[:, [1, 2, 3, 5, 6, 7]] 

train_loader, val_loader, test_loader = create_dataloaders(
    x_train, y_train, x_val, y_val, x_test
)

Xử lý trùng lặp nội bộ...
 - Loại bỏ 99 hàng trùng lặp trong Train
 - Loại bỏ 110 hàng trùng lặp trong Val
Xử lý trùng lặp giữa Train và Val...
 - Loại bỏ 187 chuỗi Train bị trùng với Val
Xử lý lỗi ngày tháng...
 - Phát hiện 1010 hàng có lỗi ngày trong Train
 - Phát hiện 749 hàng có lỗi ngày trong Val
Xây dựng từ điển và map index...
 - Kích thước từ điển 934 (PAD=0, UNK=1)
Sinh thủ công thêm chuỗi...
 - Train tăng từ 54714 lên 105245 mẫu


In [19]:
model = AdvancedTARNet(vocab_size=VOCAB_SIZE).to(DEVICE)

optimizer = optim.AdamW(
    model.parameters(), 
    lr=LEARNING_RATE, 
    weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=MAX_LR, 
    epochs=NUM_EPOCHS, 
    steps_per_epoch=len(train_loader), 
    pct_start=PCT_START,
    final_div_factor=FINAL_DIV_FACTOR
)

In [20]:
train_model(model, train_loader, val_loader, y_val_orig, optimizer, scheduler)

Epoch 01/100 | LR: 0.000046 | Train Loss: 1.5592 | Val Score: 17.3675
  => Mô hình tốt nhất! (Val Score: 17.3675)
Epoch 02/100 | LR: 0.000064 | Train Loss: 1.7866 | Val Score: 17.1965
  => Mô hình tốt nhất! (Val Score: 17.1965)
Epoch 03/100 | LR: 0.000092 | Train Loss: 1.9590 | Val Score: 16.9935
  => Mô hình tốt nhất! (Val Score: 16.9935)
Epoch 04/100 | LR: 0.000132 | Train Loss: 2.0491 | Val Score: 16.9297
  => Mô hình tốt nhất! (Val Score: 16.9297)
Epoch 05/100 | LR: 0.000181 | Train Loss: 1.8961 | Val Score: 16.8245
  => Mô hình tốt nhất! (Val Score: 16.8245)
Epoch 06/100 | LR: 0.000238 | Train Loss: 1.7287 | Val Score: 16.7144
  => Mô hình tốt nhất! (Val Score: 16.7144)
Epoch 07/100 | LR: 0.000302 | Train Loss: 1.6832 | Val Score: 13.2454
  => Mô hình tốt nhất! (Val Score: 13.2454)
Epoch 08/100 | LR: 0.000372 | Train Loss: 1.4581 | Val Score: 4.0075
  => Mô hình tốt nhất! (Val Score: 4.0075)
Epoch 09/100 | LR: 0.000445 | Train Loss: 1.0090 | Val Score: 2.5501
  => Mô hình tốt nhất

In [21]:
model.load_state_dict(torch.load(CHECKPOINT_FILE, map_location=DEVICE))

<All keys matched successfully>

In [22]:
val_preds = run_inference(model, val_loader)

def wmape(y_t, y_p):
    sum_abs_error = np.sum(np.abs(y_t - y_p))
    sum_abs_true = np.sum(np.abs(y_t))
    return (sum_abs_error / (sum_abs_true + 1e-8)) * 100

print("\n" + "=" * 76)
print(f"{'THUỘC TÍNH':<12} | {'MAE':<10} | {'MSE':<12} | {'RMSE':<10} | {'WMAPE':<10}")
print("-" * 76)

for i, attr in enumerate(ATTRIBUTE_LIST):
    y_t = y_val_orig[:, i]
    y_p = val_preds[:, i]
    
    mae = np.mean(np.abs(y_t - y_p))
    mse = np.mean(np.square(y_t - y_p))
    rmse = np.sqrt(mse)
    wmape_val = wmape(y_t, y_p)
    
    print(f"Attr {attr:<7} | {mae:<10.4f} | {mse:<12.4f} | {rmse:<10.4f} | {wmape_val:<10.4f}")

print("=" * 76)

overall_mae = np.mean(np.abs(y_val_orig - val_preds))
overall_mse = np.mean(np.square(y_val_orig - val_preds))
overall_rmse = np.sqrt(overall_mse)
overall_wmape = wmape(y_val_orig, val_preds)
final_wmse = evaluate_wmse(y_val_orig, val_preds)

print(f"{'MAE':<15}: {overall_mae:.4f}")
print(f"{'MSE':<15}: {overall_mse:.4f}")
print(f"{'RMSE':<15}: {overall_rmse:.4f}")
print(f"{'WMAPE':<15}: {overall_wmape:.4f}")
print(f"{'WMSE':<15}: {final_wmse:.4f}")
print("=" * 76)


THUỘC TÍNH   | MAE        | MSE          | RMSE       | WMAPE     
----------------------------------------------------------------------------
Attr 1       | 0.8870     | 2.4298       | 1.5588     | 13.2469   
Attr 2       | 1.7361     | 10.8215      | 3.2896     | 13.6195   
Attr 3       | 1.7308     | 34.7926      | 5.8985     | 3.4829    
Attr 4       | 0.9881     | 3.0119       | 1.7355     | 14.7625   
Attr 5       | 2.2979     | 18.1420      | 4.2593     | 17.2096   
Attr 6       | 1.9543     | 42.8076      | 6.5428     | 3.9267    
MAE            : 1.5990
MSE            : 18.6676
RMSE           : 4.3206
WMAPE          : 6.9046
WMSE           : 0.8597


In [23]:
test_preds = run_inference(model, test_loader)
submission_df = pd.DataFrame({"id": x_test["id"]})

for idx, attr in enumerate(ATTRIBUTE_LIST):
    submission_df[f"attr_{attr}"] = test_preds[:, idx].astype(np.uint16)

submission_df.to_csv(SUBMISSION_FILE, index=False)
print(submission_df)
print(submission_df.dtypes)

           id  attr_1  attr_2  attr_3  attr_4  attr_5  attr_6
0       n6r61       2       7      48       9      24      71
1       jkctd       7      16      64      11      19      65
2       a4060       5      10      40       2       1      71
3       fkbud      10      24       4       4       6      20
4       ui5gu       9      28      29       5      19      53
...       ...     ...     ...     ...     ...     ...     ...
108668  phk1w       8       8      67       7      14      13
108669  sgb1f       7      16      89       6      22      91
108670  e3s26       6      20      59       7      15      95
108671  d0kwv       7      23      40      11      20      50
108672  xfvdy       2      24      44      12      27      19

[108673 rows x 7 columns]
id        object
attr_1    uint16
attr_2    uint16
attr_3    uint16
attr_4    uint16
attr_5    uint16
attr_6    uint16
dtype: object
